# Trial Thresholds — grid T x K GraphRAG 2-Stage, dan GraphRAG 1-Stage di tiap T

Notebook ini mencari **threshold tabel (T) dan kolom (K)**. Preprocessing 2-Stage diatur dengan **saklar on/off** di bagian 3; defaultnya preprocessing `config.py` (poin 30). GraphRAG 1-Stage (nama tabel saja, poin 39) dijalankan di tiap T sebagai pembanding. Sampel 20% dev set per tingkat kesulitan (stratified).

| Bagian | Isi |
|---|---|
| 1-4 | Setup, import, konfigurasi (saklar preprocessing, grid), data |
| 5 | Index embedding |
| 6 | **Grid**: SLA, kolom, dan token tiap (T, K), dan 1-Stage tiap T |
| 7 | **EX** tiap baris (zero-shot, butuh GPU) |
| 8 | **2-Stage vs 1-Stage di T yang sama**, lalu threshold dipilih **manual** oleh peneliti (`BEST_T` / `BEST_K`) |
| 9 | Bandingkan dua baris per query |
| 10 | Export Excel (`Trial Thresholds`) |

Semua kombinasi preprocessing ada di `trial_preprocessing.ipynb`. Kode bersama kedua notebook ada di `src/experiments/trial_steps.py`.

> Notebook ini hanya menjalankan eksperimen. Threshold dipilih peneliti dari tabel bagian 6-8, lalu ditulis ke `config.py` (poin 33).

## 1. Setup

In [1]:
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/william-darma-wijaya/thesis-2803.git"
REPO_BRANCH = "main"

def sh(cmd: str) -> None:
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True)

def is_repo_root(p: Path) -> bool:
    return (p / "src" / "core" / "config.py").is_file()

if not is_repo_root(Path.cwd()):
    if is_repo_root(Path.cwd().parent):      # dijalankan dari notebooks/
        os.chdir("..")
    else:                                    # Kaggle: clone dulu
        if not is_repo_root(Path("thesis-2803").resolve()):
            sh(f"git clone -b {REPO_BRANCH} {REPO_URL}")
        else:                                # sudah pernah di-clone: pastikan di branch yang benar
            sh(f"git -C thesis-2803 fetch origin {REPO_BRANCH} && git -C thesis-2803 checkout {REPO_BRANCH} && git -C thesis-2803 pull origin {REPO_BRANCH}")
        os.chdir("thesis-2803")
REPO_ROOT = Path.cwd()
assert is_repo_root(REPO_ROOT), f"{REPO_ROOT} bukan root repo thesis-2803"
print("Repo root:", REPO_ROOT)
print("Branch   :", subprocess.run("git rev-parse --abbrev-ref HEAD", shell=True, capture_output=True, text=True).stdout.strip())

# nltk dibutuhkan parser SQL Spider (process_sql.py); sentence-transformers untuk BGE-M3
sh("pip install -q -U 'bitsandbytes>=0.46.1' sentence-transformers transformers nltk openpyxl")
import nltk
for pkg in ("punkt", "punkt_tab"):
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"[WARN] nltk.download({pkg}) gagal: {e}")

$ git clone -b main https://github.com/william-darma-wijaya/thesis-2803.git


Cloning into 'thesis-2803'...


Repo root: /kaggle/working/thesis-2803
Branch   : main
$ pip install -q -U 'bitsandbytes>=0.46.1' sentence-transformers transformers nltk openpyxl


## 2. Import

In [2]:
import sys, json, time, shutil, subprocess
from collections import Counter
from dataclasses import replace

import numpy as np
import pandas as pd
import networkx as nx
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from sentence_transformers import util
from tqdm.auto import tqdm

sys.path.insert(0, str(REPO_ROOT))

from src.core.config import PipelineConfig
from src.core.schema import normalize_identifier
from src.retrieval.retrieval import (
    retrieve_candidate_tables, retrieve_graphrag_schema, column_nodes_to_schema, _extract_phrases, k_from_pct,
    value_linked_columns, trace_schema_paths, traversal_graph,
)
from src.retrieval.value_linking import get_value_index
from src.experiments import trial_steps as ts
from src.experiments import result_tables as rt
from src.experiments.result_tables import show
from src.metrics.sla import compute_sla, aggregate_sla
from src.utils.sampling import distribution_table

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)
print("Imports OK | CUDA:", torch.cuda.is_available())

Imports OK | CUDA: True


## 3. Konfigurasi

**Saklar preprocessing 2-Stage** (`True` = nyala). Default = preprocessing `config.py` (poin 30). Kalau saklar diubah, notebook mencetak bedanya dengan `config.py`; preprocessing yang dipakai pipeline tetap `config.py` sampai diubah di sana.

| Saklar | `True` | `False` |
|---|---|---|
| `STOPWORD_FILTER` | buang n-gram yang diawali/diakhiri stopword (`edge`) | semua n-gram (`none`) |
| `VALUE_LINKING` | kolom yang nilainya disebut persis di pertanyaan ikut jadi anchor | mati |
| `VALUE_NUMBERS` | value linking juga untuk angka/tahun (hanya kalau `VALUE_LINKING = True`) | teks saja |
| `FK_ONLY` | dari jalur traversal hanya kolom JOIN (`fk_only`) | semua node di jalur (`full`) |
| `CONNECT_NO_PK` | kolom tabel tanpa PK disambung di graph traversal | tidak |
| `NATURAL_NAMES` | embed nama natural Spider (`student id`) | nama asli (`stuid`) |

`EXTRA_PREPROC` menambah preprocessing lain ke grid yang sama: tiap entri hanya menyebut saklar yang berbeda dari di atas. Kosong = hanya satu preprocessing.

| Knob lain | Arti |
|---|---|
| `GRID_T`, `GRID_K` | threshold yang dicoba. Tiap (T, K) x preprocessing = satu run; 1-Stage hanya punya T |
| `RUN_ONE_STAGE` | 1-Stage di tiap T (pembanding bagian 8) |
| `RUN_EX` | EX zero-shot tiap baris (bagian 7, butuh GPU) |
| `TARGET_P`, `TARGET_R` | baris yang memenuhi keduanya diberi warna hijau (penanda visual saja) |

In [3]:
SAMPLE_RATIO = 0.20        # 20% per tingkat kesulitan, sama dengan ablation few-shot
MAX_NGRAM = 2              # bigram (config.py)

# --- Saklar preprocessing GraphRAG 2-Stage (True = nyala); default = config.py ---
STOPWORD_FILTER = True     # True = edge, False = none
VALUE_LINKING   = True     # value linking teks
VALUE_NUMBERS   = True    # value linking juga untuk angka/tahun
FK_ONLY         = True     # True = fk_only, False = full
CONNECT_NO_PK   = True     # sambung tabel tanpa PK di graph traversal
NATURAL_NAMES   = True     # embed nama natural Spider
# Preprocessing tambahan di grid yang sama; tiap entri = saklar yang berbeda dari di atas
EXTRA_PREPROC = []         # contoh: [dict(STOPWORD_FILTER=False, VALUE_NUMBERS=True)]

GRID_T = (0.50, 0.55, 0.60, 0.65, 0.70)    # threshold tabel (Stage 1), 2-Stage dan 1-Stage
GRID_K = (0.55, 0.60, 0.65, 0.70, 0.75)    # threshold kolom (Stage 2), hanya 2-Stage
RUN_ONE_STAGE = True       # 1-Stage (nama tabel saja) di tiap T
EXCLUDE_EMPTY_GOLD = False
RUN_EX = True              # bagian 7: EX zero-shot tiap baris (butuh GPU)
TARGET_P, TARGET_R = 0.50, 0.90

cfg = PipelineConfig(top_k_tables_pct=0, top_k_columns_pct=0, max_ngram=MAX_NGRAM)
cfg.data_path = ts.find_data_path(cfg, REPO_ROOT)
_switches = dict(STOPWORD_FILTER=STOPWORD_FILTER, VALUE_LINKING=VALUE_LINKING, VALUE_NUMBERS=VALUE_NUMBERS,
                 FK_ONLY=FK_ONLY, CONNECT_NO_PK=CONNECT_NO_PK, NATURAL_NAMES=NATURAL_NAMES)
PREPROC = [ts.switches_to_settings(**{k.lower(): v for k, v in {**_switches, **extra}.items()})
           for extra in [{}] + EXTRA_PREPROC]
_config_py = ts.settings_of(cfg)
print("Data path:", cfg.data_path)
for i, s in enumerate(PREPROC):
    diff = {k: v for k, v in s.items() if v != _config_py[k]}
    print(f"Preprocessing {i}: {ts.describe(s)}" + (f"  (beda dengan config.py: {diff})" if diff else "  (= config.py)"))
n_runs = len(PREPROC) * len(GRID_T) * len(GRID_K) + (len(GRID_T) if RUN_ONE_STAGE else 0)
print(f"{n_runs} baris di grid")

Data path: /kaggle/input/datasets/alrette/spiderdataset/spider_data
Preprocessing 0: stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural  (beda dengan config.py: {'value_link': 'text + numbers'})
30 baris di grid


## 4. Load data, graph, sampel dev set, dan gold schema

In [4]:
data = ts.load_trial_data(cfg, SAMPLE_RATIO, EXCLUDE_EMPTY_GOLD)
# Nama pendek untuk sel EDA dan analisis di bawah
schema_df, graph, db_schema, samples, sample_dbs = data.schema_df, data.graph, data.db_schema, data.samples, data.sample_dbs
print(f"Dev set: {len(data.dev_data)} query | sampel: {len(data.indices)} | dipakai: {len(samples)} "
      f"| gold gagal di-parse: {data.n_failed} | gold kosong: {data.n_empty}")
print(f"Database di sampel: {len(sample_dbs)} dari {schema_df['Database'].nunique()}")
display(pd.DataFrame(distribution_table(data.dev_data, data.indices)).style.format(
    {"full_share": "{:.1%}", "sample_share": "{:.1%}"}).set_caption("Distribusi difficulty: dev penuh vs sampel"))

07:36:02 | INFO     | src.core.schema | Graph built: 4503 nodes, 3847 edges
07:36:03 | INFO     | src.retrieval.baseline | Table graph built: 876 table-nodes, 744 FK-edges
07:36:03 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
07:36:03 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-Coder-7B-Instruct "HTTP/1.1 200 OK"
07:36:03 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-7B-Instruct/resolve/c03e6d358207e414f1eca0bb1891e29f1db0e242/config.json "HTTP/1.1 307 Temporary Redirect"
07:36:03 | WARNING  | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
07:36:03 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-7B-Instruct/c03e6d358207e414f1eca0bb1891e29f1db0e242/config.json?%2FQwen%2FQwen2.5-C

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

07:36:03 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-7B-Instruct/resolve/c03e6d358207e414f1eca0bb1891e29f1db0e242/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
07:36:03 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-7B-Instruct/c03e6d358207e414f1eca0bb1891e29f1db0e242/tokenizer_config.json?%2FQwen%2FQwen2.5-Coder-7B-Instruct%2Fresolve%2Fc03e6d358207e414f1eca0bb1891e29f1db0e242%2Ftokenizer_config.json=&etag=%22acee076f49bf3c0298e15de0909d1da7b392f0c3%22 "HTTP/1.1 200 OK"
07:36:03 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-7B-Instruct/c03e6d358207e414f1eca0bb1891e29f1db0e242/tokenizer_config.json?%2FQwen%2FQwen2.5-Coder-7B-Instruct%2Fresolve%2Fc03e6d358207e414f1eca0bb1891e29f1db0e242%2Ftokenizer_config.json=&etag=%22acee076f49bf3c0298e15de0909d1da7b392f0c3%22 "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

07:36:03 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-Coder-7B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
07:36:03 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-Coder-7B-Instruct/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
07:36:03 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-7B-Instruct/resolve/c03e6d358207e414f1eca0bb1891e29f1db0e242/vocab.json "HTTP/1.1 307 Temporary Redirect"
07:36:03 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-7B-Instruct/c03e6d358207e414f1eca0bb1891e29f1db0e242/vocab.json?%2FQwen%2FQwen2.5-Coder-7B-Instruct%2Fresolve%2Fc03e6d358207e414f1eca0bb1891e29f1db0e242%2Fvocab.json=&etag=%224783fe10ac3adce15ac8f358ef5462739852c569%22 "HTTP/1.1 200 OK"
07:36:03 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/res

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

07:36:04 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-7B-Instruct/resolve/c03e6d358207e414f1eca0bb1891e29f1db0e242/added_tokens.json "HTTP/1.1 404 Not Found"
07:36:04 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-7B-Instruct/resolve/c03e6d358207e414f1eca0bb1891e29f1db0e242/special_tokens_map.json "HTTP/1.1 404 Not Found"
07:36:04 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-7B-Instruct/resolve/c03e6d358207e414f1eca0bb1891e29f1db0e242/chat_template.jinja "HTTP/1.1 404 Not Found"
07:36:04 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-Coder-7B-Instruct "HTTP/1.1 200 OK"


Dev set: 1034 query | sampel: 207 | dipakai: 207 | gold gagal di-parse: 0 | gold kosong: 7
Database di sampel: 19 dari 166


,difficulty,full_n,full_share,sample_n,sample_share
0,easy,248,24.0%,50,24.2%
1,medium,446,43.1%,89,43.0%
2,hard,174,16.8%,35,16.9%
3,extra,166,16.1%,33,15.9%


## 5. Bangun index embedding

In [5]:
data = ts.load_embedder(data)
for db in tqdm(sample_dbs, desc="Index"):
    for names in dict.fromkeys(s["nama_embed"] for s in PREPROC):
        ts.index_for(data, db, replace(cfg, **ts.knobs_of({"nama_embed": names})))
    if RUN_ONE_STAGE:
        ts.table_index_for(data, db, embed_columns=False)
print("Selesai.")

07:36:04 | INFO     | sentence_transformers.base.model | No device provided, using cuda:0
07:36:04 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3 "HTTP/1.1 200 OK"
07:36:04 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/5617a9f61b028005a4858fdac845db406aefb181/modules.json "HTTP/1.1 307 Temporary Redirect"
07:36:04 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/modules.json?%2FBAAI%2Fbge-m3%2Fresolve%2F5617a9f61b028005a4858fdac845db406aefb181%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
07:36:04 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/modules.json?%2FBAAI%2Fbge-m3%2Fresolve%2F5617a9f61b028005a4858fdac845db406aefb181%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

07:36:04 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/5617a9f61b028005a4858fdac845db406aefb181/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
07:36:04 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config_sentence_transformers.json?%2FBAAI%2Fbge-m3%2Fresolve%2F5617a9f61b028005a4858fdac845db406aefb181%2Fconfig_sentence_transformers.json=&etag=%221fba91c78a6c8e17227058ab6d4d3acb5d8630a9%22 "HTTP/1.1 200 OK"
07:36:04 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config_sentence_transformers.json?%2FBAAI%2Fbge-m3%2Fresolve%2F5617a9f61b028005a4858fdac845db406aefb181%2Fconfig_sentence_transformers.json=&etag=%221fba91c78a6c8e17227058ab6d4d3acb5d8630a9%22 "HTTP/1.1 200 OK"


config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

07:36:04 | INFO     | sentence_transformers.base.model | Loading SentenceTransformer model from BAAI/bge-m3.
07:36:05 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/5617a9f61b028005a4858fdac845db406aefb181/README.md "HTTP/1.1 307 Temporary Redirect"
07:36:05 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/README.md?%2FBAAI%2Fbge-m3%2Fresolve%2F5617a9f61b028005a4858fdac845db406aefb181%2FREADME.md=&etag=%22e5a320176edd6ee1cfb256e68ee5ac0004de9447%22 "HTTP/1.1 200 OK"
07:36:05 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/README.md?%2FBAAI%2Fbge-m3%2Fresolve%2F5617a9f61b028005a4858fdac845db406aefb181%2FREADME.md=&etag=%22e5a320176edd6ee1cfb256e68ee5ac0004de9447%22 "HTTP/1.1 200 OK"


README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

... (80 outputs left out)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Selesai.


## 6. Grid: SLA, kolom, dan token

Satu baris = satu preprocessing di satu (T, K), atau 1-Stage di satu T. Makin rendah K: recall naik, precision turun. Rekap lengkap tersimpan di `outputs/tables/trial_thresholds_recap.csv` dan sheet Excel `Trial Thresholds`; tabel di bawah hanya kolom inti. Arti tiap kolom: `context/TRIAL_COLUMNS_EXPLAINED.md`.

In [6]:
TRIALS = [ts.two_stage_trial(cfg, s, t, k) for s in PREPROC for t in GRID_T for k in GRID_K]
if RUN_ONE_STAGE:
    TRIALS += [ts.one_stage_trial(cfg, t) for t in GRID_T]
TRIAL_CACHE = globals().get("TRIAL_CACHE", {})     # hasil per query yang sudah dihitung, dipakai ulang kalau sel dijalankan lagi
recap, RESULTS = ts.run_trials(data, TRIALS, TRIAL_CACHE, progress=lambda x: tqdm(x, desc="Grid"))
print("Rekap ->", ts.save_recap(recap, REPO_ROOT / "outputs" / "tables", ts.THRESHOLDS_RECAP))
print(f"n = {len(samples)} query | MAX_NGRAM = {cfg.max_ngram}")
display(ts.styled_recap(recap, TARGET_P, TARGET_R))

Grid:   0%|          | 0/30 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

... (200 outputs left out)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Rekap -> /kaggle/working/thesis-2803/outputs/tables/trial_thresholds_recap.csv
n = 207 query | MAX_NGRAM = 2


,pipeline,table_threshold,column_threshold,stopword,value_link,traversal,no_pk,nama_embed,col_P_ne,col_R_ne,tbl_R,cols_sent,tables_pct,cols_pct,schema_tokens,prompt_tokens_k0,prompt_red_db
trial,,,,,,,,,,,,,,,,,
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.50/K0.55",graphrag,0.50,0.55,edge,text + numbers,fk_only,connected,natural,18.8%,99.4%,96.6%,19.5,94.2%,79.9%,176.9,364.7,8.9%
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.50/K0.60",graphrag,0.50,0.60,edge,text + numbers,fk_only,connected,natural,23.5%,98.3%,96.6%,15.3,87.8%,62.8%,148.2,336.0,16.0%
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.50/K0.65",graphrag,0.50,0.65,edge,text + numbers,fk_only,connected,natural,31.5%,96.5%,96.1%,11.2,82.4%,47.2%,122.0,309.8,22.6%
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.50/K0.70",graphrag,0.50,0.70,edge,text + numbers,fk_only,connected,natural,42.4%,95.1%,95.7%,8.4,73.6%,36.7%,98.3,286.1,28.5%
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.50/K0.75",graphrag,0.50,0.75,edge,text + numbers,fk_only,connected,natural,50.3%,90.1%,93.2%,6.3,62.3%,28.5%,72.4,260.2,35.0%
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.55/K0.55",graphrag,0.55,0.55,edge,text + numbers,fk_only,connected,natural,19.3%,98.9%,96.1%,18.8,91.4%,77.8%,170.5,358.3,10.5%
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.55/K0.60",graphrag,0.55,0.60,edge,text + numbers,fk_only,connected,natural,24.3%,97.8%,96.1%,14.8,84.9%,61.2%,142.6,330.4,17.4%
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.55/K0.65",graphrag,0.55,0.65,edge,text + numbers,fk_only,connected,natural,32.3%,96.5%,96.1%,10.9,80.0%,46.4%,118.3,306.1,23.5%
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.55/K0.70",graphrag,0.55,0.70,edge,text + numbers,fk_only,connected,natural,42.8%,95.1%,95.7%,8.4,72.4%,36.4%,97.4,285.2,28.7%


## 7. EX per baris (zero-shot, butuh GPU)

- **Zero-shot:** k belum dipilih pada tahap ini, dan tanpa contoh few-shot beda EX hanya berasal dari beda schema. Prompt-nya sama persis dengan `prompt_tokens_k0`, jadi EX dan token di satu baris berasal dari prompt yang sama. Yang dibaca adalah **perbandingan antar konfigurasi**, bukan EX akhir.
- **Prompt yang identik hanya digenerate sekali** (greedy, deterministik) dan disimpan di `outputs/tables/sla_trial_ex_cache.jsonl`, dipakai bersama kedua notebook trial. Jalankan ulang sel ini untuk melanjutkan run yang terputus; di Kaggle, unduh file cache itu supaya tidak hilang.
- **Tidak memilih apa pun** (poin 33). EX dihitung di dev, jadi ikut dalam limitation poin 34.

In [7]:
if RUN_EX:
    recap, llm = ts.score_ex(recap, RESULTS, cfg, data.tokenizer, REPO_ROOT / ts.EX_CACHE, globals().get("llm"),
                             progress=lambda x: tqdm(x, desc="EX"))
    ts.save_recap(recap, REPO_ROOT / "outputs" / "tables", ts.THRESHOLDS_RECAP)
display(ts.styled_recap(recap, TARGET_P, TARGET_R))

07:37:22 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-Coder-7B-Instruct "HTTP/1.1 200 OK"
07:37:22 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-Coder-7B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
07:37:22 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-Coder-7B-Instruct/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
07:37:22 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-Coder-7B-Instruct "HTTP/1.1 200 OK"
07:37:22 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-Coder-7B-Instruct "HTTP/1.1 200 OK"
07:37:22 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-7B-Instruct/resolve/c03e6d358207e414f1eca0bb1891e29f1db0e242/adapter_config.json "HTTP/1.1 404 Not Found"
07:37:23 | INFO     | httpx | HTTP Req

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

07:37:23 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-Coder-7B-Instruct/tree/c03e6d358207e414f1eca0bb1891e29f1db0e242?recursive=true&expand=false "HTTP/1.1 200 OK"


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

07:37:23 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-Coder-7B-Instruct/xet-read-token/c03e6d358207e414f1eca0bb1891e29f1db0e242 "HTTP/1.1 200 OK"


... (5 outputs left out)

EX:   0%|          | 0/30 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Digenerate: 2170 | dari cache: 4040 | gagal (dihitung salah): 0


,pipeline,table_threshold,column_threshold,stopword,value_link,traversal,no_pk,nama_embed,ex,col_P_ne,col_R_ne,tbl_R,cols_sent,tables_pct,cols_pct,schema_tokens,prompt_tokens_k0,prompt_red_db
trial,,,,,,,,,,,,,,,,,,
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.50/K0.55",graphrag,0.50,0.55,edge,text + numbers,fk_only,connected,natural,67.1%,18.8%,99.4%,96.6%,19.5,94.2%,79.9%,176.9,364.7,8.9%
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.50/K0.60",graphrag,0.50,0.60,edge,text + numbers,fk_only,connected,natural,67.6%,23.5%,98.3%,96.6%,15.3,87.8%,62.8%,148.2,336.0,16.0%
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.50/K0.65",graphrag,0.50,0.65,edge,text + numbers,fk_only,connected,natural,65.7%,31.5%,96.5%,96.1%,11.2,82.4%,47.2%,122.0,309.8,22.6%
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.50/K0.70",graphrag,0.50,0.70,edge,text + numbers,fk_only,connected,natural,64.7%,42.4%,95.1%,95.7%,8.4,73.6%,36.7%,98.3,286.1,28.5%
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.50/K0.75",graphrag,0.50,0.75,edge,text + numbers,fk_only,connected,natural,63.8%,50.3%,90.1%,93.2%,6.3,62.3%,28.5%,72.4,260.2,35.0%
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.55/K0.55",graphrag,0.55,0.55,edge,text + numbers,fk_only,connected,natural,67.1%,19.3%,98.9%,96.1%,18.8,91.4%,77.8%,170.5,358.3,10.5%
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.55/K0.60",graphrag,0.55,0.60,edge,text + numbers,fk_only,connected,natural,68.1%,24.3%,97.8%,96.1%,14.8,84.9%,61.2%,142.6,330.4,17.4%
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.55/K0.65",graphrag,0.55,0.65,edge,text + numbers,fk_only,connected,natural,65.2%,32.3%,96.5%,96.1%,10.9,80.0%,46.4%,118.3,306.1,23.5%
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.55/K0.70",graphrag,0.55,0.70,edge,text + numbers,fk_only,connected,natural,64.7%,42.8%,95.1%,95.7%,8.4,72.4%,36.4%,97.4,285.2,28.7%


## 8. GraphRAG 2-Stage vs GraphRAG 1-Stage di T yang sama, lalu pilih threshold (manual)

Sel ini **tidak memilih** threshold. Ia menyusun bahan keputusan, dan peneliti mengisi `BEST_T` / `BEST_K` di sel berikutnya.

Perbandingan apple-to-apple (IMPLEMENTATION_DECISIONS.md poin 33):
- **T sama untuk kedua pipeline.** GraphRAG 1-Stage tidak punya threshold kolom, jadi setiap baris GraphRAG 2-Stage `(T, K)` dibandingkan dengan **satu** baris `GraphRAG 1-Stage | T` yang sama. Mengganti K hanya mengubah sisi GraphRAG 2-Stage.
- **Query sama** (sampel yang sama), jadi reduksi = `1 - mean(GraphRAG 2-Stage) / mean(GraphRAG 1-Stage)` = `1 - Σ GraphRAG 2-Stage / Σ GraphRAG 1-Stage`.
- Tiga ukuran, dari yang paling dekat ke retrieval sampai yang dibayar LLM: **kolom** di prompt, **token schema** (DDL saja), **token prompt** (zero-shot). Reduksi token prompt selalu lebih kecil dari reduksi token schema, karena bagian non-schema (task, pertanyaan, instruksi, dan nanti few-shot) sama besar di kedua pipeline.
- **Dekomposisi kolom**: `table_effect` = 1 - kolom tabel GraphRAG 2-Stage kalau dikirim utuh / kolom GraphRAG 1-Stage (beda pilihan tabel); `prune_effect` = 1 - kolom GraphRAG 2-Stage / kolom tabel GraphRAG 2-Stage utuh (pemangkasan kolom). `(1 - total) = (1 - table_effect) x (1 - prune_effect)`.
- SLA dan EX (bagian 7, kalau `RUN_EX = True`) ditampilkan di samping supaya penghematan dibaca bersama recall dan EX yang dikorbankan.
- Perbandingan tiap baris dengan DDL penuh database-nya ada di rekap (kolom `*_db`, sheet `Trial Thresholds`).

In [8]:
vs = ts.compare_with_one_stage(recap)
vs.to_csv(REPO_ROOT / "outputs" / "tables" / "trial_thresholds_vs_1stage.csv", float_format="%.4f")
_pct = [c for c in ["tbl_R", "col_P_ne", "col_R_ne", "col_R_ne_B", "cols_red", "schema_tok_red", "prompt_tok_red",
                    "table_effect", "prune_effect", "ex_G", "ex_B", "ex_delta"] if c in vs.columns]
display(vs.style.format({**{c: "{:.1%}" for c in _pct}, "table_threshold": "{:.2f}", "column_threshold": "{:.2f}",
                         **{c: "{:.1f}" for c in vs.columns if c.endswith(("_G", "_B")) and c not in _pct}}))
print("*_G = 2-Stage, *_B = 1-Stage di T yang sama, *_red = 1 - G/B (positif = 2-Stage lebih hemat). Index = preprocessing + T/K")

,table_threshold,column_threshold,tbl_R,col_P_ne,col_R_ne,col_R_ne_B,cols_G,cols_B,cols_red,schema_tok_G,schema_tok_B,schema_tok_red,prompt_tok_G,prompt_tok_B,prompt_tok_red,table_effect,prune_effect,ex_G,ex_B,ex_delta
trial,,,,,,,,,,,,,,,,,,,,
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.50/K0.55",0.50,0.55,96.6%,18.8%,99.4%,100.0%,19.5,25.0,22.1%,176.9,209.6,15.6%,364.7,397.4,8.2%,1.7%,20.7%,67.1%,69.1%,-1.9%
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.50/K0.60",0.50,0.60,96.6%,23.5%,98.3%,100.0%,15.3,25.0,39.0%,148.2,209.6,29.3%,336.0,397.4,15.4%,5.5%,35.5%,67.6%,69.1%,-1.4%
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.50/K0.65",0.50,0.65,96.1%,31.5%,96.5%,100.0%,11.2,25.0,55.4%,122.0,209.6,41.8%,309.8,397.4,22.0%,13.0%,48.8%,65.7%,69.1%,-3.4%
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.50/K0.70",0.50,0.70,95.7%,42.4%,95.1%,100.0%,8.4,25.0,66.3%,98.3,209.6,53.1%,286.1,397.4,28.0%,23.2%,56.2%,64.7%,69.1%,-4.3%
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.50/K0.75",0.50,0.75,93.2%,50.3%,90.1%,100.0%,6.3,25.0,74.8%,72.4,209.6,65.5%,260.2,397.4,34.5%,33.9%,61.9%,63.8%,69.1%,-5.3%
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.55/K0.55",0.55,0.55,96.1%,19.3%,98.9%,99.5%,18.8,24.1,21.8%,170.5,201.7,15.4%,358.3,389.4,8.0%,1.2%,20.8%,67.1%,70.0%,-2.9%
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.55/K0.60",0.55,0.60,96.1%,24.3%,97.8%,99.5%,14.8,24.1,38.7%,142.6,201.7,29.3%,330.4,389.4,15.2%,5.0%,35.4%,68.1%,70.0%,-1.9%
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.55/K0.65",0.55,0.65,96.1%,32.3%,96.5%,99.5%,10.9,24.1,54.7%,118.3,201.7,41.3%,306.1,389.4,21.4%,11.6%,48.7%,65.2%,70.0%,-4.8%
"2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.55/K0.70",0.55,0.70,95.7%,42.8%,95.1%,99.5%,8.4,24.1,65.2%,97.4,201.7,51.7%,285.2,389.4,26.8%,20.7%,56.2%,64.7%,70.0%,-5.3%


*_G = 2-Stage, *_B = 1-Stage di T yang sama, *_red = 1 - G/B (positif = 2-Stage lebih hemat). Index = preprocessing + T/K


In [9]:
# PILIHAN PENELITI: isi setelah membaca tabel bagian 6-8. None = tetap pakai threshold config.py.
BEST_T = None        # threshold tabel, dipakai 2-Stage DAN 1-Stage
BEST_K = None        # threshold kolom, hanya 2-Stage
BEST_PREPROC = 0     # nomor preprocessing yang dicetak di bagian 3 (0 = saklar utama)

_out = REPO_ROOT / "outputs" / "tables"
if BEST_T is not None and BEST_K is not None:
    settings = PREPROC[BEST_PREPROC]
    trial = ts.two_stage_trial(cfg, settings, BEST_T, BEST_K)[0]
    rec = {"table_similarity_threshold": BEST_T, "semantic_similarity_threshold": BEST_K, "preprocessing": settings,
           "preprocessing_knobs": ts.knobs_of(settings), "rule": "dipilih manual oleh peneliti", "n": len(samples),
           "max_ngram": cfg.max_ngram}
    if trial in vs.index:
        rec.update({k: float(v) for k, v in vs.loc[trial].items() if k not in ("table_threshold", "column_threshold")})
    (_out / "best_thresholds.json").write_text(json.dumps(rec, indent=2), encoding="utf-8")
    print(f"Dipilih: T = {BEST_T:.2f}, K = {BEST_K:.2f}, {ts.describe(settings)} -> {_out / 'best_thresholds.json'}\n"
          "Tulis ke src/core/config.py (atau isi di eval_pipeline.ipynb / ablation_fewshot.ipynb):\n"
          f"    table_similarity_threshold: float = {BEST_T:.2f}\n    semantic_similarity_threshold: float = {BEST_K:.2f}")
    if settings != ts.settings_of(cfg):
        print(f"Preprocessing ini beda dengan config.py (poin 30): ubah juga {ts.knobs_of(settings)}")
else:
    print(f"BEST_T / BEST_K belum diisi (config.py: T = {cfg.table_similarity_threshold:.2f}, K = {cfg.semantic_similarity_threshold:.2f})")

BEST_T / BEST_K belum diisi (config.py: T = 0.60, K = 0.70)


## 9. Bandingkan dua baris per query

Isi `A` dan `B` dengan nama baris (index tabel rekap). Hasilnya: kolom yang ditambahkan B (dan kolom gold yang ikut masuk), serta kolom yang dibuang B (dan kolom gold yang ikut hilang).

In [10]:
names = list(recap.index)
A, B = names[0], names[1]
d = ts.compare_trials(RESULTS, A, B)
print(f"{A}\n  ->  {B}\nquery berubah: {len(d)} / {len(samples)} | gold masuk: {(d['gold_gained'].str.len() > 0).sum()} "
      f"| gold hilang: {(d['gold_lost'].str.len() > 0).sum()}")
display(d[d["gold_gained"].str.len() > 0][["qid", "db_id", "question", "gold_gained", "added"]].head(10))
display(d[d["gold_lost"].str.len() > 0][["qid", "db_id", "question", "gold_lost", "removed"]].head(10))

2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.50/K0.55
  ->  2-stage | stopword=edge, value_link=text + numbers, traversal=fk_only, no_pk=connected, nama_embed=natural | T0.50/K0.60
query berubah: 190 / 207 | gold masuk: 0 | gold hilang: 6


,qid,db_id,question,gold_gained,added


,qid,db_id,question,gold_lost,removed
30,148,car_1,What is the number of car models created by the car maker American Motor Company?,[car_makers.fullname],"[car_makers.fullname, continents.contid, continents.continent, countries.continent, co..."
119,642,tvshow,What are the ids of all tv channels that have more than 2 TV channels?,[tv_channel.country],"[cartoon.original_air_date, cartoon.title, cartoon.written_by, tv_channel.content, tv_..."
145,757,world_1,What is the language that is used by the largest number of Asian nations?,[country.continent],"[city.district, country.continent, country.headofstate, country.localname, country.sur..."
170,885,network_1,What are the names of the high schoolers and how many friends does each have?,[friend.student_id],"[friend.student_id, likes.student_id]"
175,904,network_1,"Show the names of high schoolers who have likes, and numbers of likes for each.",[likes.student_id],"[friend.friend_id, friend.student_id, likes.student_id]"
176,905,network_1,"What are the names of high schoolers who have likes, and how many likes does each have?",[likes.student_id],"[friend.friend_id, friend.student_id, likes.student_id]"


## 10. Export ke Excel

Sheet `Trial Thresholds`. Kalau `trial_preprocessing_recap.csv` juga ada di folder yang sama, sheet `Trial Preprocessing` ikut. Di Kaggle, salinannya ditaruh di `/kaggle/working/`, bersama cache EX.

In [11]:
XLSX = REPO_ROOT / "outputs" / "tables" / "hasil_trial_thresholds.xlsx"
subprocess.run([sys.executable, "src/experiments/export_excel.py", "--skip-missing",
                "--tables-dir", str(REPO_ROOT / "outputs" / "tables"), "--out", str(XLSX)], check=True)
print("Excel:", XLSX)
if Path("/kaggle/working").exists():
    shutil.copy(XLSX, "/kaggle/working/hasil_trial_thresholds.xlsx")
    shutil.copy(REPO_ROOT / ts.EX_CACHE, "/kaggle/working/sla_trial_ex_cache.jsonl") if (REPO_ROOT / ts.EX_CACHE).exists() else None
    print("Salinan untuk diunduh: /kaggle/working/hasil_trial_thresholds.xlsx (dan cache EX)")

Excel tersimpan: /kaggle/working/thesis-2803/outputs/tables/hasil_trial_thresholds.xlsx
Excel: /kaggle/working/thesis-2803/outputs/tables/hasil_trial_thresholds.xlsx
Salinan untuk diunduh: /kaggle/working/hasil_trial_thresholds.xlsx (dan cache EX)
